In [ ]:
# 1. Authenticate Google Drive first, so the run does not stop later waiting for login
options(repos = c(CRAN = "https://cloud.r-project.org"))
for (p in c("remotes", "googledrive")) if (!requireNamespace(p, quietly = TRUE)) install.packages(p)
options(gargle_oauth_cache = "~/.cache/gargle")   # keep the token for this Colab session
googledrive::drive_auth(email = TRUE)
cat("Drive user:", googledrive::drive_user()$emailAddress, "\n")

# Installs sihlib and microdatasus (both from GitHub)
remotes::install_github("lucasolt/sihlib", upgrade = "never")
library(sihlib)
library(dplyr)

In [ ]:
# 2. Parameters: the only cell that usually needs editing
UFS         <- c("RS", "SC")             # SIH-RD is published by the HOSPITAL's state
COMP_INI    <- as.Date("2025-01-01")     # first month (competencia) to load
COMP_FIM    <- Sys.Date()                # last month; unpublished months are skipped
DATA_INI    <- as.Date("2025-01-01")     # admissions STARTED in this period
DATA_FIM    <- as.Date("2025-12-31")
CID         <- "F"                       # prefixes, or data.frame(grupo, de, ate)
ESPEC       <- c("05", "87")             # bed specialty; NULL = all
GAP         <- 1                         # AIH starting up to GAP days after the previous = same stay
ANOS        <- 2025
DIR_CACHE   <- "sih_databases"           # local copy of the cache
PASTA_DRIVE <- "sih_databases"           # Drive cache folder; NULL = no Drive
REBAIXAR    <- character()               # months to download again, e.g. c("2025_11")

# Population by municipality (6-digit IBGE code): data.frame(municipio, pop) or with a column ano
POP <- NULL   # e.g. read.csv("pop.csv", colClasses = c(municipio = "character"))

In [ ]:
# 3. Pipeline
base <- sih_carregar(UFS, COMP_INI, COMP_FIM, dir_cache = DIR_CACHE,
                     pasta_drive = PASTA_DRIVE, rebaixar = REBAIXAR)
print(attr(base, "log"))

# Filter by CID and specialty only; the period is applied after reconstruction
# so that chains of AIHs are not cut at the limits
int <- base |>
  sih_filtrar(cid = CID, espec = ESPEC) |>
  reconstruir_internacoes(gap = GAP) |>
  filter(ini >= DATA_INI, ini <= DATA_FIM)
cat(nrow(int), "stays;", sum(int$emendado), "with chained AIHs\n")

cont <- contar_municipio(int, por = "residencia", anos = ANOS)
if (!is.null(POP)) {
  taxa <- taxa_municipio(cont, POP, por = 10000)
  head(taxa[order(-taxa$razao_oe), ])
} else head(cont)